<a href="https://colab.research.google.com/github/VakitiSriVarsha/Gen-AI/blob/main/Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q openai faiss-cpu tiktoken pypdf gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 23.3 MB/s eta 0:00:00


In [4]:
import os
import numpy as np
import faiss
import tiktoken
import gradio as gr
from pypdf import PdfReader
from openai import OpenAI
from google.colab import userdata

In [5]:
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI()

In [6]:
pdf_text = """
TransformersAttention Is All You Need are a class of deep learning architectures introduced in the paper
''. They rely on self-attention mechanisms instead of recurrence.

Self-attention allows tokens to attend to other tokens in the sequence and compute
context-aware representations using queries, keys, and values.

Multi-head attention improves performance by allowing the model to focus on different
parts of the sequence simultaneously.

Transformers require positional encodings because attention alone does not capture order.

Popular transformer-based models include BERT, GPT, and T5.
"""

In [7]:
def chunk_text(text, chunk_size=200, overlap=40):
    tokenizer = tiktoken.get_encoding("cl100k_base")
    tokens = tokenizer.encode(text)

    chunks = []
    start = 0

    while start < len(tokens):
        end = start + chunk_size
        chunk = tokenizer.decode(tokens[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

In [8]:
def embed_texts(texts):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return [item.embedding for item in response.data]

In [9]:
class FaissVectorStore:
    def __init__(self, dim):
        self.index = faiss.IndexFlatL2(dim)
        self.texts = []

    def add(self, embeddings, texts):
        self.index.add(np.array(embeddings).astype("float32"))
        self.texts.extend(texts)

    def search(self, query_embedding, top_k=3):
        k = min(top_k, len(self.texts))
        _, idx = self.index.search(
            np.array([query_embedding]).astype("float32"), k
        )
        return [self.texts[i] for i in idx[0]]

In [10]:
def retrieve(query, vector_store, top_k=3):
    query_embedding = embed_texts([query])[0]
    return vector_store.search(query_embedding, top_k)

In [11]:
chunks = chunk_text(pdf_text)
embeddings = embed_texts(chunks)

store = FaissVectorStore(len(embeddings[0]))
store.add(embeddings, chunks)

print("✅ Vector store ready")
print("Chunks:", len(chunks))

✅ Vector store ready
Chunks: 1


In [12]:
def build_prompt(context_chunks, question):
    context = "\n\n".join(context_chunks)

    return f"""
Answer the question ONLY using the context below.
If the answer is not present, say "I do not know."

Context:
{context}

Question:
{question}

Answer:
"""

In [13]:
def generate_answer(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content

In [14]:
def rag_pipeline(question):
    retrieved_chunks = retrieve(question, store)
    prompt = build_prompt(retrieved_chunks, question)
    answer = generate_answer(prompt)
    return answer, retrieved_chunks

In [15]:
def chatbot(question):
    answer, sources = rag_pipeline(question)
    sources_text = "\n\n---\n\n".join(sources)
    return f"ANSWER:\n{answer}\n\nSOURCES:\n{sources_text}"

gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(placeholder="Ask a question from the document..."),
    outputs="text",
    title="📄 RAG Q&A Chatbot",
    description="Retrieval-Augmented Generation using FAISS + OpenAI"
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://20a9ad19f5559b6e48.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
